# System Design for AI: A Real-Time Voice Agent, Millisecond by Millisecond

The runnable code of [System Design for AI: A Real-Time Voice Agent, Millisecond by Millisecond](https://residual-stream.lvtan71.workers.dev/posts/voice-agent-system-design/), in the order the post runs it. Each
code cell is followed by the output the post printed, so you can compare.

[![Open in Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/lvtan71/residual-stream-code/blob/main/notebooks/voice-agent-system-design.ipynb)

**Hardware:** Runs on a CPU; Colab's free runtime is enough. About 1 min end to end.


The post's numbers come from an Apple M4 laptop. Other hardware and library
versions can change the last digits of timings and of some floating-point
results; the conclusions should not change.

In [ ]:
# Setup: fetch the helpers and data this post's code uses into a working folder.
import os, shutil, subprocess, sys
SLUG = "voice-agent-system-design"
ROOT = os.path.abspath("..") if os.path.isdir("../shared") else os.path.abspath("residual-stream-code")
if not os.path.isdir(ROOT):
    subprocess.run(["git", "clone", "-q", "--depth", "1", "https://github.com/lvtan71/residual-stream-code", ROOT], check=True)
WORK = os.path.join(ROOT, "work", SLUG)
os.makedirs(WORK, exist_ok=True)
for f in []:
    shutil.copy(os.path.join(ROOT, f), WORK)
    if f.endswith(".tar.gz"):                       # the blog's posts, as of that run
        shutil.unpack_archive(os.path.join(WORK, os.path.basename(f)), WORK)
os.chdir(WORK)
sys.path.insert(0, WORK)
print("working in", WORK)

## The latency budget

[Read this part in the post](https://residual-stream.lvtan71.workers.dev/posts/voice-agent-system-design/#the-latency-budget)

In [ ]:
# A cascaded voice agent's response time, stage by stage: from the moment
# the user stops speaking to the moment they hear the first syllable of the
# reply. Every figure is an assumption for a well-built system, stated so it
# can be changed, not a measurement.
stages = [  # (stage, milliseconds, why)
    ("audio frame + jitter buffer", 40, "20 ms Opus frames; a jitter buffer of about one frame"),
    ("network, user -> server",     30, "one way, same region, WebRTC over UDP"),
    ("end-of-turn detection",      400, "VAD waits for this much silence before deciding the user is done"),
    ("speech-to-text, final",      100, "a streaming recognizer has the words already; it finalizes"),
    ("LLM time to first token",    250, "prefill of a cached 2,000-token prompt plus queueing"),
    ("LLM, first clause",          150, "TTS needs a few words to start: ~8 tokens at ~19 ms each"),
    ("text-to-speech, first audio", 120, "streaming TTS, time to its first audio chunk"),
    ("network, server -> user",     30, "one way"),
    ("playback buffer",             40, "the client's own jitter buffer before audio plays"),
]
total = sum(ms for _, ms, _ in stages)
print(f"{'stage':30s} {'ms':>5s} {'cumulative':>11s}")
run = 0
for name, ms, _ in stages:
    run += ms
    print(f"{name:30s} {ms:5d} {run:11d}")
print(f"\ntotal {total} ms; end-of-turn detection alone is {400 / total:.0%} of it")

# Two levers, applied to the same budget.
for name, change in (("semantic turn detection (400 -> 150 ms of silence)", {"end-of-turn detection": 150}),
                     ("speech-to-speech model (STT + LLM + TTS -> 350 ms)", None)):
    if change:
        t = sum(change.get(n, ms) for n, ms, _ in stages)
    else:
        t = sum(ms for n, ms, _ in stages if not n.startswith(("speech-to-text", "LLM", "text-to-speech"))) + 350
    print(f"  with {name}: {t} ms")

The post printed:

```text
stage                             ms  cumulative
audio frame + jitter buffer       40          40
network, user -> server           30          70
end-of-turn detection            400         470
speech-to-text, final            100         570
LLM time to first token          250         820
LLM, first clause                150         970
text-to-speech, first audio      120        1090
network, server -> user           30        1120
playback buffer                   40        1160

total 1160 ms; end-of-turn detection alone is 34% of it
  with semantic turn detection (400 -> 150 ms of silence): 910 ms
  with speech-to-speech model (STT + LLM + TTS -> 350 ms): 890 ms
```

## Capacity: 1,000 calls

[Read this part in the post](https://residual-stream.lvtan71.workers.dev/posts/voice-agent-system-design/#capacity-1000-calls)

In [ ]:
import math
import numpy as np

# How many GPUs does the LLM stage of 1,000 simultaneous voice calls need?
# An idealized simulation of continuous batching ("Serving Metrics and Prefill–Decode Disaggregation"), one iteration
# at a time, for an 8B model in BF16 on H100s (80 GB).
#   Calls: each sends the LLM a request every ~20 s on average (the caller
#   talks for a few seconds; the agent speaks its ~45-word answer for ~17 s);
#   arrivals are random (Poisson), 50 per second in all.
#   A request: a 2,000-token conversation so far plus 150 new tokens, and
#   60 output tokens. Each call is pinned to one GPU, so its cache is there.
#   An iteration decodes one token for every active request and prefills the
#   requests admitted this iteration. Its time is the larger of reading the
#   weights and every KV cache it touches at 3.35 TB/s, and the arithmetic at
#   40% of 989 TFLOPS.
# Where the 2,000 tokens of each call's history live between turns decides
# everything (2,000 tokens x 128 KiB = 262 MB per call, 262 GB for 1,000):
#   "HBM"     : kept in GPU memory, so only the 150 new tokens are prefilled,
#               but every call's cache must fit beside the weights;
#   "re-prefill": nothing kept, all 2,150 tokens prefilled every turn;
#   "CPU"     : kept in host memory and copied back over PCIe (~25 GB/s
#               effective) when the call's next request arrives.
N, W, BW, FLOPS, HBM = 8e9, 16e9, 3.35e12, 0.4 * 989e12, 80e9
KV, CONTEXT, NEW, OUT, MAX_BATCH, PCIE = 131072, 2000, 150, 60, 200, 25e9
CALLS, TURN_S = 1000, 20.0

def simulate(gpus, mode, seconds=120, seed=0):
    calls_per_gpu = math.ceil(CALLS / gpus)
    if mode == "HBM" and W + calls_per_gpu * CONTEXT * KV + MAX_BATCH * (NEW + OUT) * KV > 0.9 * HBM:
        return None                                          # the caches don't fit
    prefill = NEW if mode != "re-prefill" else CONTEXT + NEW
    reload = CONTEXT * KV / PCIE if mode == "CPU" else 0.0  # copy the history back before prefill
    rng = np.random.default_rng(seed)
    t, arrivals = 0.0, []
    while t < seconds:
        t += rng.exponential(TURN_S / CALLS)
        arrivals.append((t + reload, int(rng.integers(CALLS))))  # (ready time, call)
    arrivals.sort(reverse=True)
    clock, waiting, active = [0.0] * gpus, [[] for _ in range(gpus)], [[] for _ in range(gpus)]
    ttft, itl = [], []
    while arrivals or any(waiting) or any(active):
        g = min(range(gpus), key=lambda i: clock[i])
        now = clock[g]
        while arrivals and arrivals[-1][0] <= now:
            ready, call = arrivals.pop()
            waiting[call % gpus].append(ready - reload)       # sticky: a call always goes to its GPU
        if not waiting[g] and not active[g]:
            nxt = [a for a, c in arrivals if c % gpus == g]
            clock[g] = min(nxt) if nxt else float("inf")
            if all(c == float("inf") for c in clock):
                break
            continue
        admit = waiting[g][: max(0, MAX_BATCH - len(active[g]))]
        waiting[g] = waiting[g][len(admit):]
        reads = W + (len(active[g]) + len(admit)) * CONTEXT * KV        # weights + every cache touched
        dt = max(reads / BW, 2 * N * (len(active[g]) + prefill * len(admit)) / FLOPS)
        clock[g] = now + dt
        if active[g]:
            itl.append(dt)
        active[g] = [r - 1 for r in active[g] if r > 1] + [OUT - 1] * len(admit)
        ttft += [clock[g] - a for a in admit]
    warm = len(ttft) // 10
    return np.array(ttft[warm:]), np.array(itl)

print(f"{CALLS} calls, a request every {TURN_S:.0f} s per call; 2,000 + 150 prompt tokens, {OUT} out")
print(f"{'history kept in':>15s} {'GPUs':>4s} {'TTFT p50':>9s} {'TTFT p99':>9s} {'ITL p50':>8s} {'ITL p99':>8s}")
for mode in ("HBM", "re-prefill", "CPU"):
    for gpus in (1, 2, 4, 6, 8, 12):
        r = simulate(gpus, mode)
        if r is None:
            print(f"{mode:>15s} {gpus:4d}   {1000 // gpus + 1} calls' caches don't fit in 80 GB")
            continue
        ttft, itl = r
        if np.median(ttft) > 10:
            print(f"{mode:>15s} {gpus:4d}   overloaded: requests queue without bound")
            continue
        print(f"{mode:>15s} {gpus:4d} {np.median(ttft) * 1e3:6.0f} ms {np.percentile(ttft, 99) * 1e3:6.0f} ms "
              f"{np.median(itl) * 1e3:5.1f} ms {np.percentile(itl, 99) * 1e3:5.1f} ms", flush=True)

The post printed:

```text
1000 calls, a request every 20 s per call; 2,000 + 150 prompt tokens, 60 out
history kept in GPUs  TTFT p50  TTFT p99  ITL p50  ITL p99
            HBM    1   1001 calls' caches don't fit in 80 GB
            HBM    2   501 calls' caches don't fit in 80 GB
            HBM    4   251 calls' caches don't fit in 80 GB
            HBM    6      9 ms     16 ms   5.0 ms   6.2 ms
            HBM    8      8 ms     16 ms   4.9 ms   6.2 ms
            HBM   12      8 ms     14 ms   4.9 ms   6.1 ms
     re-prefill    1   overloaded: requests queue without bound
     re-prefill    2   overloaded: requests queue without bound
     re-prefill    4   8715 ms  22437 ms  20.4 ms 2440.8 ms
     re-prefill    6    194 ms   1321 ms   5.3 ms 261.2 ms
     re-prefill    8    106 ms    554 ms   5.0 ms  87.9 ms
     re-prefill   12     90 ms    349 ms   4.9 ms  87.1 ms
            CPU    1     22 ms     41 ms   6.5 ms  13.3 ms
            CPU    2     20 ms     31 ms   5.5 ms   6.8 ms
            CPU    4     19 ms     27 ms   5.1 ms   6.3 ms
            CPU    6     19 ms     27 ms   5.0 ms   6.2 ms
            CPU    8     19 ms     26 ms   4.9 ms   6.2 ms
            CPU   12     18 ms     24 ms   4.9 ms   6.1 ms
```